In [1]:
import json
import os
import time
import re
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEV_PATH  = "project/data/dev.jsonl"
TEST_PATH = "project/data/test_30.jsonl"

RUN_TEST = True  

SPLITS = {
    "dev": DEV_PATH,
    **({"test": TEST_PATH} if RUN_TEST else {})
}

Path("outputs").mkdir(exist_ok=True, parents=True)
Path("project/results").mkdir(exist_ok=True, parents=True)

ZERO_SHOT_PROMPT = """You are a cautious healthcare assistant. Answer accurately and conservatively.
Return ONLY valid JSON with keys: short_answer, confidence_level, clinical_notes.
confidence_level must be one of: high, medium, low.
No markdown, no code fences, no extra text.

Question: {question}
JSON:
"""

PROMPT_PATH = "project/results/zero_shot_prompt.txt"
with open(PROMPT_PATH, "w", encoding="utf-8") as f:
    f.write(ZERO_SHOT_PROMPT)

print("Setup OK")
print("Splits:", list(SPLITS.keys()))
print("Prompt saved to:", PROMPT_PATH)


Setup OK
Splits: ['dev', 'test']
Prompt saved to: project/results/zero_shot_prompt.txt


In [2]:
import torch
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,     
    offload_folder="offload",      # cria pasta para offload em disco
    low_cpu_mem_usage=True,
)
model.eval()

gen_cfg = GenerationConfig(
    max_new_tokens=800,            
    do_sample=False,
    temperature=0.0,
    use_cache=True,
)

print("CUDA available:", torch.cuda.is_available())
print("Model device (first param):", next(model.parameters()).device)



`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-19 12:23:09.024679: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768825389.047953     507 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768825389.057654     507 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-19 12:23:09.250528: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CUDA available: True
Model device (first param): cuda:0


In [3]:
def load_questions(path: str):
    qs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("question") or obj.get("instruction")
            if q and isinstance(q, str):
                qs.append(q.strip())
    return qs

def extract_any_json(text: str):
    # extrai o primeiro JSON bem formado 
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
    return None


In [4]:
SYSTEM_MSG = (
    "You are a cautious healthcare assistant. "
    "Return ONLY a valid JSON object with keys: short_answer, confidence_level, clinical_notes. "
    "confidence_level must be one of: high, medium, low. "
    "No markdown, no code fences, no extra text."
)

def generate_zero_shot(question: str):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": ZERO_SHOT_PROMPT.format(question=question)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    prompt = prompt + "{"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            generation_config=gen_cfg,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    dt = time.time() - t0

    full_text = tokenizer.decode(out[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    gen_text = full_text[len(prompt_text):].strip()

    obj = extract_any_json(gen_text)
    if obj is None:
        obj = {
            "short_answer": "",
            "confidence_level": "low",
            "clinical_notes": "Model output did not contain valid JSON."
        }

    return obj, gen_text, dt


In [5]:
def run_split(split_name: str, in_path: str, method_name: str = "zero"):
    questions = load_questions(in_path)
    n = len(questions)

    if n == 0:
        raise ValueError(f"Não carreguei perguntas de {in_path}")

    out_path = f"outputs/{split_name}_zero_shot.jsonl"
    results = []

    print(f"\n=== RUN {method_name.upper()} | split={split_name} | n={n} ===")
    print("Input:", in_path)
    print("Output:", out_path)

    t_global0 = time.time()
    times = []

    for i, q in enumerate(questions, start=1):
        t_start = time.time()
        print(f"[{split_name}] {i}/{n} → starting inference at {time.strftime('%H:%M:%S')}", flush=True)

        obj, raw_text, dt = generate_zero_shot(q)

        t_end = time.time()
        print(
            f"[{split_name}] {i}/{n} ← finished in {t_end - t_start:.2f}s "
            f"(model time {dt:.2f}s)",
            flush=True
        )

        times.append(dt)

        avg = sum(times) / len(times)
        eta = avg * (n - i)

        results.append({
            "split": split_name,
            "method": method_name,
            "question": q,
            "parsed_json": obj,
            "raw_text": raw_text,
            "timing_sec": dt
        })

        # log a cada pergunta 
        print(f"[{split_name}] {i}/{n} | {dt:.2f}s | avg {avg:.2f}s | ETA {eta/60:.1f} min")

    # guardar
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    total = time.time() - t_global0
    print(f"Saved: {out_path} | total {total/60:.1f} min | avg {sum(times)/len(times):.2f}s/question")

    return out_path

# Corre DEV sempre; TEST só se RUN_TEST=True
generated_files = []
for split_name, in_path in SPLITS.items():
    generated_files.append(run_split(split_name, in_path, method_name="zero"))




=== RUN ZERO | split=dev | n=30 ===
Input: project/data/dev.jsonl
Output: outputs/dev_zero_shot.jsonl
[dev] 1/30 → starting inference at 12:25:09
[dev] 1/30 ← finished in 3.30s (model time 3.24s)
[dev] 1/30 | 3.24s | avg 3.24s | ETA 1.6 min
[dev] 2/30 → starting inference at 12:25:13
[dev] 2/30 ← finished in 3.35s (model time 3.34s)
[dev] 2/30 | 3.34s | avg 3.29s | ETA 1.5 min
[dev] 3/30 → starting inference at 12:25:16
[dev] 3/30 ← finished in 2.38s (model time 2.38s)
[dev] 3/30 | 2.38s | avg 2.99s | ETA 1.3 min
[dev] 4/30 → starting inference at 12:25:18
[dev] 4/30 ← finished in 2.00s (model time 2.00s)
[dev] 4/30 | 2.00s | avg 2.74s | ETA 1.2 min
[dev] 5/30 → starting inference at 12:25:20
[dev] 5/30 ← finished in 3.56s (model time 3.56s)
[dev] 5/30 | 3.56s | avg 2.90s | ETA 1.2 min
[dev] 6/30 → starting inference at 12:25:24
[dev] 6/30 ← finished in 3.31s (model time 3.31s)
[dev] 6/30 | 3.31s | avg 2.97s | ETA 1.2 min
[dev] 7/30 → starting inference at 12:25:27
[dev] 7/30 ← finish

In [6]:
import re
import json

SYSTEM_MSG = (
    "You are a cautious healthcare assistant. "
    "Return ONLY a valid JSON object with keys: short_answer, confidence_level, clinical_notes. "
    "confidence_level must be one of: high, medium, low. "
    "No markdown, no code fences, no extra text."
)

def extract_any_json(text: str):
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
    return None

def generate_one(question: str):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    prompt = prompt + "{"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    full_text = tokenizer.decode(out[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    text = full_text[len(prompt_text):].strip()

    obj = extract_any_json(text)
    if obj is None:
        obj = {
            "short_answer": "",
            "confidence_level": "low",
            "clinical_notes": "Model output did not contain valid JSON."
        }

    return obj, text



In [7]:
import json

for path in generated_files:
    n_lines = sum(1 for _ in open(path, "r", encoding="utf-8"))
    first = json.loads(open(path, "r", encoding="utf-8").readline())

    print("\nFILE:", path)
    print("Lines:", n_lines)
    print("Sample keys:", list(first.keys()))
    print("Sample parsed_json:", first.get("parsed_json"))



FILE: outputs/dev_zero_shot.jsonl
Lines: 30
Sample keys: ['split', 'method', 'question', 'parsed_json', 'raw_text', 'timing_sec']
Sample parsed_json: {'short_answer': '', 'confidence_level': 'low', 'clinical_notes': 'Model output did not contain valid JSON.'}

FILE: outputs/test_zero_shot.jsonl
Lines: 30
Sample keys: ['split', 'method', 'question', 'parsed_json', 'raw_text', 'timing_sec']
Sample parsed_json: {'short_answer': '', 'confidence_level': 'low', 'clinical_notes': 'Model output did not contain valid JSON.'}


In [8]:
import json
from pathlib import Path

def validate_output(path: str):
    rows = [json.loads(l) for l in open(path, "r", encoding="utf-8") if l.strip()]
    required = {"short_answer", "confidence_level", "clinical_notes"}
    allowed_conf = {"high", "medium", "low"}

    missing = 0
    bad_conf = 0
    empty_sa = 0

    for r in rows:
        pj = r.get("parsed_json", {})
        if not isinstance(pj, dict) or not required.issubset(set(pj.keys())):
            missing += 1
            continue
        conf = str(pj.get("confidence_level", "")).strip().lower()
        if conf not in allowed_conf:
            bad_conf += 1
        if str(pj.get("short_answer", "")).strip() == "":
            empty_sa += 1

    print("\nVALIDATION:", path)
    print("Rows:", len(rows))
    print("Missing required keys:", missing)
    print("Bad confidence_level:", bad_conf)
    print("Empty short_answer:", empty_sa)

for path in generated_files:
    validate_output(path)




VALIDATION: outputs/dev_zero_shot.jsonl
Rows: 30
Missing required keys: 0
Bad confidence_level: 0
Empty short_answer: 30

VALIDATION: outputs/test_zero_shot.jsonl
Rows: 30
Missing required keys: 0
Bad confidence_level: 0
Empty short_answer: 30


In [9]:
import boto3
from pathlib import Path

S3_BUCKET = "healthcare-longevity"
S3_PREFIX = "healthcare-longevity"

s3 = boto3.client("s3")

print("S3 client ready")
print("Bucket:", S3_BUCKET)
print("Prefix:", S3_PREFIX)


S3 client ready
Bucket: healthcare-longevity
Prefix: healthcare-longevity


In [10]:
def upload_if_exists(local_path: str, s3_key: str):
    p = Path(local_path)
    if not p.exists():
        print("SKIP (not found):", local_path)
        return
    s3.upload_file(str(p), S3_BUCKET, s3_key)
    print(f"UPLOADED → s3://{S3_BUCKET}/{s3_key}")

print("\n=== UPLOADING OUTPUT FILES ===")

# outputs zero-shot (dev sempre, test só se existir)
upload_if_exists(
    "outputs/dev_zero_shot.jsonl",
    f"{S3_PREFIX}/results/dev_zero_shot.jsonl"
)

upload_if_exists(
    "outputs/test_zero_shot.jsonl",
    f"{S3_PREFIX}/results/test_zero_shot.jsonl"
)

print("\n=== UPLOADING DATASETS ===")

upload_if_exists(
    "project/data/train.jsonl",
    f"{S3_PREFIX}/data/train-2.jsonl"
)

upload_if_exists(
    "project/data/test_30.jsonl",
    f"{S3_PREFIX}/data/test_30.jsonl"
)

print("\n=== UPLOADING PROMPT ===")

upload_if_exists(
    "project/results/zero_shot_prompt.txt",
    f"{S3_PREFIX}/prompts/zero_shot_prompt.txt"
)

print("\n=== S3 UPLOAD COMPLETE ===")



=== UPLOADING OUTPUT FILES ===
UPLOADED → s3://healthcare-longevity/healthcare-longevity/results/dev_zero_shot.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/results/test_zero_shot.jsonl

=== UPLOADING DATASETS ===
UPLOADED → s3://healthcare-longevity/healthcare-longevity/data/train-2.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/data/test_30.jsonl

=== UPLOADING PROMPT ===
UPLOADED → s3://healthcare-longevity/healthcare-longevity/prompts/zero_shot_prompt.txt

=== S3 UPLOAD COMPLETE ===


In [11]:
import torch
print("CUDA available:", torch.cuda.is_available())


CUDA available: True


In [12]:
print("Model device:", next(model.parameters()).device)


Model device: cuda:0


In [13]:
import json
import re
from pathlib import Path

def extract_any_json(text: str):
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
    return None

def fix_file(in_path: str, out_path: str):
    inp = Path(in_path)
    if not inp.exists():
        print("SKIP (not found):", in_path)
        return

    rows = [json.loads(l) for l in inp.open("r", encoding="utf-8") if l.strip()]
    fixed = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            text = (r.get("raw_text") or "").strip()
            if text.startswith('"'):
                text = "{" + text

            obj = extract_any_json(text)
            if obj is not None:
                r["parsed_json"] = obj
                fixed += 1

            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Wrote: {out_path} | fixed {fixed}/{len(rows)}")

fix_file("outputs/dev_zero_shot.jsonl",  "outputs/dev_zero_shot_fixed.jsonl")
fix_file("outputs/test_zero_shot.jsonl", "outputs/test_zero_shot_fixed.jsonl")


Wrote: outputs/dev_zero_shot_fixed.jsonl | fixed 30/30
Wrote: outputs/test_zero_shot_fixed.jsonl | fixed 30/30


In [14]:
import json

Q = "Em obesidade sem diabetes, qual é o racional clínico para GLP-1 e que limites de segurança devem ser comunicados?"
PATH = "outputs/dev_zero_shot_fixed2.jsonl"  # ou o ficheiro zero que estás a usar no evaluation

rows = [json.loads(l) for l in open(PATH, "r", encoding="utf-8") if l.strip()]
r = next(x for x in rows if x["question"] == Q)

print("PARSED_JSON:", r["parsed_json"])
print("\nRAW_TEXT (first 800):\n", (r.get("raw_text","")[:800]))


PARSED_JSON: {'short_answer': '', 'confidence_level': 'low', 'clinical_notes': 'Model output did not contain valid JSON.'}

RAW_TEXT (first 800):
 "short_answer": "Em obesidade sem diabetes, o GLP-1 pode auxiliar no controle do peso, mas os limites de segurança devem ser monitorados.", "confidence_level": "medium", "clinical_notes": "O GLP-1 (Glucagon-Like Peptide-1) pode ser útil na obesidade sem diabetes devido ao seu efeito saciante e na redução da fome. No entanto, é importante monitorar os efeitos colaterais, como náuseas e diarreia, e os
